In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from skfp.fingerprints import PubChemFingerprint
from mordred import Calculator, descriptors
from sklearn.svm import SVR
import joblib

def make_Pc_feature(smiles, names):
    fea_rdkit = ['qed','SPS','MaxPartialCharge','MinPartialCharge','BCUT2D_LOGPHI','BalabanJ','Kappa1','Kappa3','SMR_VSA5','TPSA','VSA_EState3','VSA_EState8','VSA_EState9','NHOHCount','MolLogP']
    rdkit2D = {}
    pubchem_fp = PubChemFingerprint()
    for i in range(len(smiles)):
        name = names[i]
        smile = smiles[i]
        mol = Chem.MolFromSmiles(smile)
        rdkit2D[name] = {}
        rdkit2D[name]['smiles'] = smile
        # MaxAbsEStateIndex
        for Descriptor, func in Descriptors.descList:
            if Descriptor in fea_rdkit:
                try:
                    rdkit2D[name][Descriptor] = func(mol)
                except:
                    pass
        # fingerprint = pubchem_fp.transform([smile])[0]
        # rdkit2D[name]['PubChemFP_30'] = fingerprint[30]
    calc = Calculator(descriptors, ignore_3D=False)
    mols = [Chem.MolFromSmiles(smile) for smile in smiles]
    mordred = calc.pandas(mols, quiet=True)
    # 只保留下面这些特征
    fea = ['ATS0se','ATS2se','ATS3se','ATS2pe','ATS0are','ATS2are','ATS5p','ATS6p','ATS0i','ATS2i','ATSC1dv','ATSC0d','ATSC1m','ATSC1v','ATSC1se','ATSC1pe','ATSC1are','ATSC1p','ATSC0i','ATSC1i','AATSC0Z','AATSC0i','BCUTdv-1l','BCUTZ-1l','BCUTpe-1l','BCUTp-1h','BCUTi-1h','BalabanJ','SpMAD_DzZ','SpMAD_Dzm','SpAbs_Dzse','SpDiam_Dzse','SpMAD_Dzse','SpDiam_Dzpe','SpMAD_Dzpe','SpDiam_Dzare','SpMAD_Dzare','SpMAD_Dzp','SpDiam_Dzi','SpMAD_Dzi','Sse','Spe','Sare','Si','Mp','SsCH3','ETA_eta','ETA_eta_L','ETA_dEpsilon_D','fMF','CIC0','ZMIC1','FilterItLogS','VMcGowan','AMID','AMID_h','MID_C','AMID_C','AMID_O','bpol','TopoPSA']
    mordred = mordred[fea]
    df_ = pd.DataFrame(rdkit2D).T
    # df_转化为表格格式，不需要index
    df_ = df_.reset_index()
    # 重命名列
    df_.columns = ['Name', 'smiles','qed','SPS','MaxPartialCharge','MinPartialCharge','BCUT2D_LOGPHI','BalabanJ','Kappa1','Kappa3','SMR_VSA5','TPSA','VSA_EState3','VSA_EState8','VSA_EState9','NHOHCount','MolLogP']
    df_ = pd.concat([df_, mordred], axis=1)
    # df_.to_csv('./exp/Pc_feature.csv', index=False)
    return df_

def predict_Pc(smiles, names = None):
    svr = joblib.load('./model/RF-Pc-2D.pkl')
    mean = pd.read_csv('./data/Pc_mean.csv', header=None)
    std = pd.read_csv('./data/Pc_std.csv', header=None)
    mean = mean.values[:76].reshape(-1)
    std = std.values[:76].reshape(-1)
    if names == None:
        names = smiles
    make_Pc_feature_ = make_Pc_feature(smiles, names)
    # make_Pc_feature_.to_excel('./exp/PC_feature.xlsx', index=False)
    make_Pc_feature_ = make_Pc_feature_.drop(columns=['Name', 'smiles'])
    # 归一化
    make_Pc_feature_ = make_Pc_feature_.values
    make_Pc_feature_ = (make_Pc_feature_ - mean) / std
    # 预测
    Tc = svr.predict(make_Pc_feature_)
    return Tc



In [2]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_A.xlsx')
smiles = df['smiles'].tolist()
pred_PC = predict_Pc(smiles)
df['PC_pred'] = np.exp(pred_PC) / 100000
df['AARD(%)'] = abs(df['PC_pred'] - df['Pc (bar)']) / df['Pc (bar)'] * 100
# 保存到文件
df.to_excel('./newexp/RF_PC_2D_TEST_A.xlsx', index=False)
from sklearn.metrics import mean_squared_error, r2_score
PC = df['PC_pred'].values
PC_pred = df['Pc (bar)'].values
r2 = r2_score(PC, PC_pred)
rmse = np.sqrt(mean_squared_error(PC, PC_pred))
mae = np.mean(abs(PC - PC_pred))
aard = np.mean(df['AARD(%)'])
print(f'Pc R^2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, AARD: {aard:.4f}')

Pc R^2: 0.9501, RMSE: 1.8505, MAE: 1.4852, AARD: 5.9490


In [3]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_AB.xlsx')
smiles = df['smiles'].tolist()
pred_PC = predict_Pc(smiles)
df['PC_pred'] = np.exp(pred_PC) / 100000
df['AARD(%)'] = abs(df['PC_pred'] - df['Pc (bar)']) / df['Pc (bar)'] * 100
# 保存到文件
df.to_excel('./newexp/RF_PC_2D_TEST_AB.xlsx', index=False)
from sklearn.metrics import mean_squared_error, r2_score
PC = df['PC_pred'].values
PC_pred = df['Pc (bar)'].values
r2 = r2_score(PC, PC_pred)
rmse = np.sqrt(mean_squared_error(PC, PC_pred))
mae = np.mean(abs(PC - PC_pred))
aard = np.mean(df['AARD(%)'])
print(f'Pc R^2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, AARD: {aard:.4f}')

Pc R^2: 0.5100, RMSE: 7.0845, MAE: 3.4161, AARD: 29.1142


In [4]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_AB.xlsx')
df2 = pd.read_excel('./newexp/TEST_A.xlsx')
# dfdrop掉df2中已有的smiles
df = df[~df['smiles'].isin(df2['smiles'])]
# 重设索引
smiles = df['smiles'].tolist()
pred_PC = predict_Pc(smiles)
df['PC_pred'] = np.exp(pred_PC) / 100000
df['AARD(%)'] = abs(df['PC_pred'] - df['Pc (bar)']) / df['Pc (bar)'] * 100
from sklearn.metrics import mean_squared_error, r2_score
PC = df['PC_pred'].values
PC_pred = df['Pc (bar)'].values
r2 = r2_score(PC, PC_pred)
rmse = np.sqrt(mean_squared_error(PC, PC_pred))
mae = np.mean(abs(PC - PC_pred))
aard = np.mean(df['AARD(%)'])
print(f'Pc R^2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, AARD: {aard:.4f}')

Pc R^2: -0.1711, RMSE: 15.2425, MAE: 10.9366, AARD: 119.3362


In [5]:
df = pd.read_excel('./homo/PURE-homo-150.xlsx')
SMILES = df['SMILES'].tolist()
names = df['name'].tolist()
pred_PC = predict_Pc(SMILES, names)
df['PC'] = np.exp(pred_PC) / 100000
df.to_csv('./homo/RF_2D_PC.csv', index=False)